# SHC-Guard v0.6 — دفتر تجربة (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alqasmhapip468-dotcom/SHC-Guard/blob/main/SHC-Guard-v0.6-Colab.ipynb)

**تجريبي، غير مدقق، لا يُستخدم لحماية بيانات حقيقية.**

هذا الدفتر يشغّل النسخة المعتمدة v0.6 من مشروع SHC-Guard (طبقة حماية حالة لجلسات مراسلة هجينة: Ed25519 + ML-DSA-65، X25519 + ML-KEM-768، XChaCha20-Poly1305) بنقرة واحدة — بلا تثبيت على هاتفك.

**كيف تشغّله:** من القائمة فوق: `Runtime ← Run all` (وقت التشغيل ← تشغيل الكل)، وانتظر. خلية بناء liboqs بطيئة (دقائق) وهذا طبيعي.

**حكم التدقيق المستقل:** REJECT — لا يُنشر للاستخدام الأمني ولا يحمي بيانات حقيقية. التفاصيل في `chatgpt-audit-v06.md` داخل المستودع. هذا الدفتر للتجربة والتعلم فقط.

المستودع: https://github.com/alqasmhapip468-dotcom/SHC-Guard

## 1) أدوات النظام

تثبيت libsodium وcmake (موجودان غالبًا في Colab — هذه الخلية للتأكيد فقط).

In [ ]:
!apt-get update -qq
!apt-get install -y -qq libsodium-dev cmake ninja-build > /dev/null 2>&1
print("system deps OK")

## 2) بناء liboqs 0.16.0 (مصغَّر — الخوارزميتان المطلوبتان فقط)

البناء الكامل عبر pip يفشل (خطأ في `src/sig/mqom`)، لذلك نبني من المصدر بخوارزميتين فقط: **ML-KEM-768** و**ML-DSA-65**، وكل العائلات الأخرى OFF — وهو البناء نفسه الذي استُخدم في حزمة v0.6. يُثبَّت في `$HOME/_oqs` لأن `liboqs-python` يبحث عنه هناك تحديدًا. هذه أبطأ خلية (دقائق) — اتركها تعمل.

In [ ]:
%%bash
set -e
cd /tmp
rm -rf liboqs liboqs-build
git clone -q --depth 1 --branch 0.16.0 https://github.com/open-quantum-safe/liboqs.git liboqs
cmake -S liboqs -B liboqs-build \
  -DCMAKE_INSTALL_PREFIX=$HOME/_oqs \
  -DCMAKE_BUILD_TYPE=Release \
  -DBUILD_SHARED_LIBS=ON \
  -DOQS_BUILD_ONLY_LIB=ON \
  -DOQS_ENABLE_KEM_BIKE=OFF \
  -DOQS_ENABLE_KEM_CLASSIC_MCELIECE=OFF \
  -DOQS_ENABLE_KEM_FRODOKEM=OFF \
  -DOQS_ENABLE_KEM_HQC=OFF \
  -DOQS_ENABLE_KEM_KYBER=OFF \
  -DOQS_ENABLE_KEM_ML_KEM=ON \
  -DOQS_ENABLE_KEM_NTRU=OFF \
  -DOQS_ENABLE_KEM_NTRUPRIME=OFF \
  -DOQS_ENABLE_KEM_ml_kem_512=OFF \
  -DOQS_ENABLE_KEM_ml_kem_768=ON \
  -DOQS_ENABLE_KEM_ml_kem_1024=OFF \
  -DOQS_ENABLE_SIG_CROSS=OFF \
  -DOQS_ENABLE_SIG_FALCON=OFF \
  -DOQS_ENABLE_SIG_MAYO=OFF \
  -DOQS_ENABLE_SIG_ML_DSA=ON \
  -DOQS_ENABLE_SIG_MQOM=OFF \
  -DOQS_ENABLE_SIG_SLH_DSA=OFF \
  -DOQS_ENABLE_SIG_SNOVA=OFF \
  -DOQS_ENABLE_SIG_STFL_LMS=OFF \
  -DOQS_ENABLE_SIG_STFL_XMSS=OFF \
  -DOQS_ENABLE_SIG_UOV=OFF \
  -DOQS_ENABLE_SIG_ml_dsa_44=OFF \
  -DOQS_ENABLE_SIG_ml_dsa_65=ON \
  -DOQS_ENABLE_SIG_ml_dsa_87=OFF
cmake --build liboqs-build --parallel 2
cmake --install liboqs-build
ls $HOME/_oqs/lib/

## 3) تثبيت مكتبات Python والتحقق

نثبّت PyNaCl وpytest وliboqs-python بالإصدارات المثبَّتة في `reference/requirements.txt`، ثم نتحقق أن الخوارزميتين الوحيدتين المتاحتين هما المطلوبتان.

In [ ]:
!pip install -q PyNaCl==1.6.2 pytest==8.3.4 liboqs-python==0.16.0.2
import oqs
print("OQS", oqs.oqs_version())
print("KEMs:", oqs.get_enabled_kem_mechanisms())
print("SIGs:", oqs.get_enabled_sig_mechanisms())
assert oqs.get_enabled_kem_mechanisms() == ('ML-KEM-768',)
assert oqs.get_enabled_sig_mechanisms() == ('ML-DSA-65',)
print("OK — الخوارزميتان المطلوبتان فقط")

## 4) تنزيل شفرة SHC-Guard v0.6

In [ ]:
!rm -rf /tmp/SHC-Guard
!git clone -q https://github.com/alqasmhapip468-dotcom/SHC-Guard.git /tmp/SHC-Guard
%cd /tmp/SHC-Guard/reference
!ls

## 5) الديمو الكامل

مصافحة هجينة بين طرفين → تحقق من البصمة خارج القناة (TOFU) → رسائل في الاتجاهين → نقطة تفتيش مسجلة في المرساة قبل إرسالها → تحدي SUSPECT يعود إلى ACTIVE. المتوقع في النهاية: `demo OK`.

In [ ]:
!python demo.py --allow-level-b --suspect-demo

## 6) الاختبارات — المتوقع: 121 ناجحة، صفر فاشلة

In [ ]:
!python -m pytest tests -q

## 7) جرّب بنفسك — عدّل الرسائل وشغّل

غيّر النصين أدناه وأعد تشغيل الخلية: مصافحة جديدة، رسالة من كل طرف، ونقطة تفتيش. كل شيء يعمل داخل هذه الخلية فقط (جلسة مؤقتة تُحذف بعدها).

In [ ]:
import sys, tempfile
sys.path.insert(0, ".")
from demo import build_pair
from shc_guard.anchor import LocalAnchor

with tempfile.TemporaryDirectory() as td:
    anchor = LocalAnchor(td + "/anchor.log")
    a, b = build_pair(allow_level_b=True, anchor=anchor)

    # رسالة من A إلى B — بدّل النص كما تشاء
    w = a.send_app("سلام من A — هذه أول رسالة".encode())
    print("B got:", b.receive(w)["status"], b.delivered[-1])

    # رسالة من B إلى A
    w = b.send_app("وعليكم السلام من B".encode())
    print("A got:", a.receive(w)["status"], a.delivered[-1])

    # نقطة تفتيش موقعة (تُسجَّل في المرساة قبل إرسالها)
    print("checkpoint:", b.receive(a.send_checkpoint()))
    print("states:", a.state, b.state, "| anchor entries:", len(anchor))

## 8) اكتب رسالتك المشفرة

الفكرة: لا تكتب نصًا مشفرًا بيدك — **التشفير يحدث داخل الجلسة**. بعد المصافحة والتحقق من البصمة، تستدعي `send_app` بنصك فيعيد لك **البايتات السلكية المشفرة** (XChaCha20-Poly1305). هذه البايتات هي رسالتك المشفرة، ولا يفكها إلا الطرف الآخر في الجلسة نفسها عبر `receive`.

بدّل سطر `MESSAGE` أدناه فقط وأعد تشغيل الخلية.

In [ ]:
# اكتب رسالتك هنا — بدّل النص فقط
MESSAGE = "هذه رسالتي السرية الأولى"

import sys, tempfile
sys.path.insert(0, ".")
from demo import build_pair
from shc_guard.anchor import LocalAnchor

with tempfile.TemporaryDirectory() as td:
    anchor = LocalAnchor(td + "/anchor.log")
    a, b = build_pair(allow_level_b=True, anchor=anchor)

    # التشفير: send_app يشفّر رسالتك داخل الجلسة ويعيد البايتات السلكية
    wire = a.send_app(MESSAGE.encode())
    print("الرسالة المشفرة (بايتات سلكية، hex):")
    print(wire.hex())
    print()
    # فك التشفير عند الطرف الآخر في الجلسة نفسها
    result = b.receive(wire)
    print("حالة الاستلام:", result["status"])
    print("الرسالة بعد فك التشفير عند B:", b.delivered[-1])

## ماذا رأيت؟ وماذا لا يعني؟

- رأيت مصافحة هجينة حقيقية (X25519 + ML-KEM-768، وهويات Ed25519 + ML-DSA-65)، ورسائل مشفرة بـ XChaCha20-Poly1305، ونقاط تفتيش موقعة، وتحدي SUSPECT يعيد الجلسة إلى ACTIVE.
- هذا **لا يعني** أن النظام آمن: المرساة هنا سجل ملفي محلي (بديل لا طرف ثالث موثوق)، والتراجع المتزامن للطرفين ما زال غير مكتشف، وعلم `--allow-level-b` للتجربة فقط (بلا عتاد حقيقي)، ونموذج Tamarin بلا أي lemma مثبت.
- اقرأ المواصفة الكاملة في `spec/SHC-Guard-v0.6.md`، وقرارات التصميم في `DESIGN_FREEZE_v06.md`، والتقرير في `REPORT.md` — وكلها داخل المستودع.
- لا تبنِ على هذه الشفرة أي نظام يحمي بيانات حقيقية.